# 03 — Threshold selection

The model outputs a fraud **probability**. To block or flag a transaction we need a **threshold**:
everything above it is treated as fraud.

- The threshold is chosen on **validation**, using a model trained on train only (it has never seen val).
- The chosen threshold is then applied **once** to the final model on **test** — these are the numbers for the README.

Choice criterion: **minimum total cost of errors** (missed fraud vs false alerts).

## Setup

In [1]:
import os
import sys
from pathlib import Path

import joblib
import pandas as pd

# Run from project root so relative paths (data/raw, models/) and `app` imports work
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

from app.data import load_data, split_xy
from app.features import build_features
from app.metrics import threshold_metrics
from app.models import make_xgb

## Data
Same time-based split as in notebook 02: train / val / test.

In [2]:
splits = split_xy(build_features(load_data()))

X_train, y_train = splits["train"]
X_val, y_val = splits["val"]
X_test, y_test = splits["test"]

print(f"train: {X_train.shape}, val: {X_val.shape}, test: {X_test.shape}")

train: (1142581, 10), val: (154094, 10), test: (555719, 10)


## Model for threshold selection
Trained on **train only**, so its scores on val are honest (the final model has seen val, so it can't be used here).

In [3]:
val_model = make_xgb().fit(X_train, y_train)
val_scores = val_model.predict_proba(X_val)[:, 1]  # probability of fraud

## How errors change with the threshold
- **TP** — fraud caught · **FN** — fraud missed · **FP** — false alert · **TN** — legit, correctly passed
- **recall** — share of fraud we catch · **precision** — share of alerts that are real fraud

In [6]:
# THRESHOLDS = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
THRESHOLDS = [0.01, 0.02, 0.03, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

rows = []
for t in THRESHOLDS:
    metrics = threshold_metrics(y_val, val_scores, t)
    rows.append({"threshold": t, **metrics})

sweep = pd.DataFrame(rows)
sweep

,threshold,tp,fp,fn,tn,precision,recall
0,0.01,906,4381,52,148755,0.1714,0.9457
1,0.02,883,2554,75,150582,0.2569,0.9217
2,0.03,859,1776,99,151360,0.3260,0.8967
3,0.05,812,1120,146,152016,0.4203,0.8476
4,0.10,751,554,207,152582,0.5755,0.7839
5,0.20,688,235,270,152901,0.7454,0.7182
6,0.30,652,165,306,152971,0.7980,0.6806
7,0.40,617,106,341,153030,0.8534,0.6441
8,0.50,568,64,390,153072,0.8987,0.5929
9,0.60,521,32,437,153104,0.9421,0.5438


**Observation:** _TODO — what happens to recall and FP as the threshold goes up?_

## Cost of errors
- **Missed fraud (FN)** ≈ average fraud amount from EDA (Q3): **$530**
- **False alert (FP)** — analyst review time + a customer whose payment was blocked: **$10** (*assumption, configurable*)

Best threshold = the one with the **lowest total cost**.

In [7]:
COST_FN = 530  # missed fraud: average fraud amount (EDA, Q3)
COST_FP = 10   # false alert: review + customer friction (assumption)

# Total cost of errors at each threshold
sweep["cost"] = sweep["fn"] * COST_FN + sweep["fp"] * COST_FP

# Row with the lowest total cost
best = sweep.loc[sweep["cost"].idxmin()]
best_threshold = float(best["threshold"])

print(
    f"best threshold: {best_threshold} | "
    f"recall {best['recall']:.1%} | precision {best['precision']:.1%} | "
    f"cost ${best['cost']:,.0f}"
)
sweep

best threshold: 0.02 | recall 92.2% | precision 25.7% | cost $65,290


,threshold,tp,fp,fn,tn,precision,recall,cost
0,0.01,906,4381,52,148755,0.1714,0.9457,71370
1,0.02,883,2554,75,150582,0.2569,0.9217,65290
2,0.03,859,1776,99,151360,0.3260,0.8967,70230
3,0.05,812,1120,146,152016,0.4203,0.8476,88580
4,0.10,751,554,207,152582,0.5755,0.7839,115250
5,0.20,688,235,270,152901,0.7454,0.7182,145450
6,0.30,652,165,306,152971,0.7980,0.6806,163830
7,0.40,617,106,341,153030,0.8534,0.6441,181790
8,0.50,568,64,390,153072,0.8987,0.5929,207340
9,0.60,521,32,437,153104,0.9421,0.5438,231930


**Observation:** The cheapest threshold is **0.02** (total cost $65,290 vs $207,340 at the
default 0.5 — 69% lower). It catches 92% of fraud, but only ~1 in 4 alerts is real
(precision 26%, ~40 false alerts per day). The optimum matches the theoretical value for
calibrated probabilities, C_FP / (C_FP + C_FN) = 10 / 540 ≈ 0.019. It is entirely driven by
the cost assumption: a higher price for a false alert (e.g. a lost customer) pushes the
threshold up. The threshold is a business decision; the notebook makes the trade-off explicit.

## Final result on test
The final model (trained on train + val) with the threshold chosen on val. Test is used **once**.

In [8]:
final_model = joblib.load("models/model.joblib")
test_scores = final_model.predict_proba(X_test)[:, 1]

final = threshold_metrics(y_test, test_scores, best_threshold)
final

{'tp': 1889,
 'fp': 9548,
 'fn': 256,
 'tn': 544026,
 'precision': 0.1652,
 'recall': 0.8807}

## Conclusion
On the test set (6 months the model has never seen, 2,145 frauds) at threshold 0.02:
- the model catches **1,889 frauds (recall 88%)** and misses 256;
- it raises **9,548 false alerts** (~50/day, 1.7% of legitimate transactions);
  precision is 16.5% — roughly 1 real fraud per 6 alerts.

Precision is lower than on validation (26%) because fraud is rarer in the test period
(0.39% vs 0.62%), so each threshold produces more false alerts per real fraud; recall
holds up (92% → 88%).

The cost-optimal threshold favours catching fraud because a miss is assumed to cost
53x more than a false alert. In production a two-tier setup would reduce manual work:
auto-block above a high threshold (~0.5, precision ~90% on val) and send the 0.02–0.5
band to analyst review — which is where the LLM investigator assists.